# Ranking Model Training

Stage-2 XGBoost ranker experiments with AWS Managed MLflow + Optuna HPO.

Guide: `docs/implementation-info/guides/ranking-model-training-guide.md`
Plan: `docs/implementation-info/ranking/ranking-model-training-plan.md`


## 1. Setup

### 1.1 Environment & credentials

Load `.env.local` (gitignored) for AWS credentials and MLflow/Optuna endpoints.
When AWS vars are missing, the notebook still runs locally (in-memory Optuna, local `s3/` mirror).


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

for _nb in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (_nb / "utils").is_dir():
        sys.path[:0] = [str(_nb)]
        break

from utils.config_loader import find_repo_root
from utils.ml_config import MLInfraConfig

REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env.local")
load_dotenv(REPO_ROOT / ".env")

ml_cfg = MLInfraConfig.from_env()
try:
    ml_cfg.validate_for_aws()
    aws_ready = True
except ValueError as exc:
    aws_ready = False
    print("AWS infra not fully configured (local-only mode):", exc)

print("S3 bucket:", ml_cfg.s3_bucket)
print("AWS ready:", aws_ready)
print("MLflow URI set:", bool(ml_cfg.mlflow_tracking_uri))


### 1.2 Load configs (YAML + feature paths)


In [ ]:
from utils.config_loader import load_feature_engineering_config
from utils.ranking_training_helpers import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    RANKING_FEATURE_COLUMNS,
    load_ranking_search_space_yaml,
    load_ranking_yaml,
)

repo_root = REPO_ROOT
fe_cfg = load_feature_engineering_config(repo_root=repo_root, environment="local_dev")
ranking_cfg = load_ranking_yaml(repo_root)
search_space = load_ranking_search_space_yaml(repo_root)

print("Default n_estimators:", ranking_cfg["n_estimators"])
print("scale_pos_weight:", ranking_cfg["scale_pos_weight"])
print("Optuna n_trials:", ranking_cfg["optuna"]["n_trials"])
print(
    "Feature columns:",
    len(CATEGORICAL_FEATURES),
    "cat +",
    len(NUMERIC_FEATURES),
    "num =",
    len(RANKING_FEATURE_COLUMNS),
)


### 1.3 Load features, cast dtypes, verify schema


In [ ]:
import pandas as pd
from utils.data_casting import cast_table
from utils.ranking_training_helpers import verify_ranking_schema
from utils.two_tower_training_helpers import resolve_features_path

features_path = resolve_features_path(repo_root, fe_cfg)
df = pd.read_parquet(features_path)
df = cast_table(
    df,
    section="features",
    config_path=repo_root / "configs" / "features" / "ml_types.yaml",
)
verify_ranking_schema(df)

print("Loaded rows:", len(df))
print("Snap dates:", sorted(df["snap_date"].astype(str).unique()))
print("Label balance:", df["label"].value_counts(normalize=True).round(4).to_dict())
df[["customer_id", "article_id", "snap_date", "label"] + RANKING_FEATURE_COLUMNS[:3]].head()


## 2. Infrastructure checks

### 2.1 MLflow tracking server status (AWS only)


In [ ]:
if aws_ready:
    from utils.ranking_training_helpers import mlflow_server_status

    status = mlflow_server_status(ml_cfg.mlflow_tracking_server_name, ml_cfg.aws_region)
    print("MLflow server status:", status)
else:
    print("Skipping MLflow server check — local-only mode")


### 2.2 Optuna study connectivity


In [ ]:
import optuna

optuna_cfg = ranking_cfg["optuna"]
storage = ml_cfg.optuna_storage_uri if aws_ready else None
study = optuna.create_study(
    study_name=optuna_cfg["study_name"],
    storage=storage,
    load_if_exists=bool(storage),
    direction=optuna_cfg["direction"],
)
print("Study:", study.study_name, "trials=", len(study.trials), "storage=", storage or "in-memory")


## 3. Data preparation

### 3.1 Feature schema groups

Show the 53-feature list selected from nb01/nb04 EDA and `features-eng.md`.


In [ ]:
feature_groups = {
    "categorical_item_catalog": CATEGORICAL_FEATURES[:11],
    "categorical_user_pref": CATEGORICAL_FEATURES[11:],
    "numeric_user_item": NUMERIC_FEATURES[:9],
    "numeric_user_category": NUMERIC_FEATURES[9:14],
    "numeric_item": NUMERIC_FEATURES[14:29],
    "numeric_user": NUMERIC_FEATURES[29:35],
    "numeric_context": NUMERIC_FEATURES[35:],
}
for name, cols in feature_groups.items():
    print(name + ":", len(cols), "features")
active_cat_cols = list(CATEGORICAL_FEATURES)
active_num_cols = list(NUMERIC_FEATURES)
active_feature_cols = active_cat_cols + active_num_cols


### 3.2 Temporal split (positives + negatives)

Unlike two-tower retrieval, ranking keeps both label=0 and label=1 rows per snap.


In [ ]:
from utils.ranking_training_helpers import apply_temporal_split_ranking, new_run_id

temporal = ranking_cfg["temporal_split"]
train_df, val_df, test_df = apply_temporal_split_ranking(df, temporal)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pos_rate = split["label"].mean() if len(split) else float("nan")
    n_users = split["customer_id"].nunique() if len(split) else 0
    print(name, "rows=", len(split), " users=", n_users, " pos_rate=", round(pos_rate, 4))


### 3.3 Stage splits to local S3 mirror (+ AWS when configured)


In [ ]:
from utils.ranking_training_helpers import stage_splits_local_ranking, stage_splits_s3_ranking

run_id = new_run_id()
local_s3_root = repo_root / fe_cfg["local_s3_root"]
split_paths = stage_splits_local_ranking(
    train_df,
    val_df,
    test_df,
    local_s3_root=local_s3_root,
    run_id=run_id,
)
print("Local staged:", split_paths)

split_uris = {}
if aws_ready:
    split_uris = stage_splits_s3_ranking(
        train_df,
        val_df,
        test_df,
        bucket=ml_cfg.s3_bucket,
        run_id=run_id,
        region=ml_cfg.aws_region,
    )
    print("S3 staged:", split_uris)


## 4. Baseline training — feature importance check

### 4.1 Train XGBClassifier with guide defaults; evaluate val AUC-PR


In [ ]:
from utils.ranking_training_helpers import build_xgb_classifier, prepare_feature_matrix, val_aucpr

x_train = prepare_feature_matrix(train_df, active_feature_cols, active_cat_cols)
y_train = train_df["label"].astype(int)
x_val = prepare_feature_matrix(val_df, active_feature_cols, active_cat_cols)
y_val = val_df["label"].astype(int)

baseline_model = build_xgb_classifier(ranking_cfg)
baseline_model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    verbose=False,
)
baseline_aucpr = val_aucpr(baseline_model, x_val, y_val)
print("Baseline val AUC-PR:", round(baseline_aucpr, 4))


### 4.2 Plot gain-based feature importance (all active features)


In [ ]:
import matplotlib.pyplot as plt
from utils.ranking_training_helpers import extract_gain_importance

baseline_importance = extract_gain_importance(baseline_model, active_feature_cols)
display(baseline_importance.head(15))

fig, ax = plt.subplots(figsize=(8, 10))
plot_df = baseline_importance.sort_values("gain", ascending=True).tail(53)
ax.barh(plot_df["feature"], plot_df["gain"])
ax.set_title("Baseline gain-based feature importance")
ax.set_xlabel("gain")
plt.tight_layout()
plt.show()


### 4.3 Drop zero-importance features before HPO


In [ ]:
from utils.ranking_training_helpers import zero_importance_features

drop_cols = zero_importance_features(baseline_importance)
print("Dropping", len(drop_cols), "zero-gain features:", drop_cols)

active_cat_cols = [c for c in active_cat_cols if c not in drop_cols]
active_num_cols = [c for c in active_num_cols if c not in drop_cols]
active_feature_cols = active_cat_cols + active_num_cols
print(
    "Active features for HPO:",
    len(active_feature_cols),
    "(",
    len(active_cat_cols),
    "cat +",
    len(active_num_cols),
    "num)",
)


### 4.4 SageMaker baseline job stub (future)

Uncomment when `pipelines/sagemaker/launch_ranking_job.py` is available.


In [ ]:
# Optional smoke: launch a single SageMaker Training Job with guide defaults.
# import subprocess
# cmd = [
#     sys.executable,
#     str(repo_root / "pipelines" / "sagemaker" / "launch_ranking_job.py"),
#     "--train-uri", split_uris.get("train", split_paths["train"]),
#     "--val-uri", split_uris.get("val", split_paths["val"]),
# ]
# print(" ".join(cmd))
# subprocess.run(cmd, check=True)


## 5. Hyperparameter optimization

### 5.1 Local Optuna trials (objective = maximize val AUC-PR; early stopping on val logloss)


In [ ]:
import mlflow
from utils.ranking_training_helpers import sample_optuna_params

if aws_ready:
    os.environ["MLFLOW_TRACKING_URI"] = ml_cfg.mlflow_tracking_uri
    mlflow.set_tracking_uri(ml_cfg.mlflow_tracking_uri)
    mlflow.set_experiment(ml_cfg.mlflow_experiment)

x_train = prepare_feature_matrix(train_df, active_feature_cols, active_cat_cols)
x_val = prepare_feature_matrix(val_df, active_feature_cols, active_cat_cols)


def ranking_objective(trial):
    trial_params = sample_optuna_params(trial, search_space)
    model = build_xgb_classifier(ranking_cfg, trial_params=trial_params)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    score = val_aucpr(model, x_val, y_val)

    if aws_ready:
        with mlflow.start_run(nested=True):
            mlflow.set_tag("model", "xgboost_ranker")
            mlflow.log_params(trial_params)
            mlflow.log_metric("val_aucpr", score)
    return score


default_trial_params = {k: ranking_cfg[k] for k in search_space if k in ranking_cfg}
if default_trial_params:
    study.enqueue_trial(default_trial_params)

n_trials = ranking_cfg["optuna"]["n_trials"]
if aws_ready:
    with mlflow.start_run(run_name="ranking_hpo_" + run_id):
        mlflow.set_tag("model", "xgboost_ranker")
        mlflow.set_tag("run_id", run_id)
        study.optimize(ranking_objective, n_trials=n_trials)
        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_val_aucpr", study.best_value)
else:
    study.optimize(ranking_objective, n_trials=n_trials)

print("Best val AUC-PR:", study.best_value)
print("Best params:", study.best_params)


### 5.2 SageMaker Processing orchestrator stub (production scale)


In [ ]:
if aws_ready:
    proc_cmd = [
        sys.executable,
        str(repo_root / "pipelines" / "sagemaker" / "hpo_processing_job.py"),
        "--train-uri",
        split_uris["train"],
        "--val-uri",
        split_uris["val"],
        "--model-type",
        "ranking",
    ]
    print(" ".join(proc_cmd))
    # subprocess.run(proc_cmd, check=True)
else:
    print("Skipping SageMaker HPO stub — local-only mode")


### 5.3 Monitor trials in MLflow


In [ ]:
if aws_ready:
    runs = mlflow.search_runs(filter_string="tags.model = 'xgboost_ranker'", max_results=20)
    metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
    param_cols = [c for c in runs.columns if c.startswith("params.")]
    display(runs[metric_cols + param_cols][:8])
else:
    print("MLflow monitoring skipped — local-only mode")


## 6. Final evaluation

### 6.1 Retrain on train+val with best params; evaluate on test


In [ ]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
x_train_val = prepare_feature_matrix(train_val_df, active_feature_cols, active_cat_cols)
y_train_val = train_val_df["label"].astype(int)
x_test = prepare_feature_matrix(test_df, active_feature_cols, active_cat_cols)
y_test = test_df["label"].astype(int)

final_model = build_xgb_classifier(ranking_cfg, trial_params=study.best_params)
final_model.fit(x_train_val, y_train_val, verbose=False)

from sklearn.metrics import average_precision_score, classification_report, roc_auc_score

test_scores = final_model.predict_proba(x_test)[:, 1]
test_aucpr = average_precision_score(y_test, test_scores)
test_rocauc = roc_auc_score(y_test, test_scores)
print("Test AUC-PR:", round(test_aucpr, 4))
print("Test ROC-AUC:", round(test_rocauc, 4))
print(classification_report(y_test, (test_scores >= 0.5).astype(int), digits=4))


### 6.2 Precision-Recall and ROC curves


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
PrecisionRecallDisplay.from_predictions(y_test, test_scores, ax=axes[0])
axes[0].set_title("Test Precision-Recall")
RocCurveDisplay.from_predictions(y_test, test_scores, ax=axes[1])
axes[1].set_title("Test ROC")
plt.tight_layout()
plt.show()


### 6.3 Oracle-candidate hit_rate@15 on test pairs


In [ ]:
from utils.ranking_training_helpers import hit_rate_at_k

hr15 = hit_rate_at_k(
    final_model,
    test_df,
    active_feature_cols,
    cat_cols=active_cat_cols,
    k=15,
)
print("Oracle hit_rate@15:", round(hr15, 4))


### 6.4 Final feature importance (top 25 by gain)


In [ ]:
final_importance = extract_gain_importance(final_model, active_feature_cols)
display(final_importance.head(25))

fig, ax = plt.subplots(figsize=(8, 8))
top25 = final_importance.head(25).sort_values("gain", ascending=True)
ax.barh(top25["feature"], top25["gain"])
ax.set_title("Final model — top-25 gain importance")
ax.set_xlabel("gain")
plt.tight_layout()
plt.show()


## 7. Results & handoff

### 7.1 MLflow runs summary


In [ ]:
if aws_ready:
    summary = mlflow.search_runs(filter_string="tags.model = 'xgboost_ranker'", max_results=50)
    print("Runs logged:", len(summary))
    display(summary.sort_values("start_time", ascending=False).head(10))
else:
    print("Local Optuna best trial value:", round(study.best_value, 4))


### 7.2 Export best params to configs/models/ranking.yaml


In [ ]:
from utils.ranking_training_helpers import export_ranking_best_params

if study.best_params:
    out_path = export_ranking_best_params(study.best_params, repo_root)
    print("Updated", out_path)


### 7.3 Save model + feature schema to s3/models/ranking/


In [ ]:
from utils.ranking_training_helpers import build_feature_schema, save_model_artifacts

feature_schema = build_feature_schema(active_cat_cols, active_num_cols)
model_dir = local_s3_root / "models" / "ranking"
artifact_paths = save_model_artifacts(final_model, feature_schema, output_dir=model_dir)
print("Saved artifacts:", artifact_paths)
